In [ ]:
# Import packages, load the study configuration, the exclusion lists and the
# location of the ANNOVAR multianno tables
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from SSc_Somatic import configure, load_config
from SSc_Somatic.burden import vcf_processing as vcf_funcs
from SSc_Somatic.loading import load_exclusion_lists
from SSc_Somatic.metadata import clean_dataframe

# Study configuration. Point this at another configuration file to analyze a different cohort.
config = configure(load_config(Path('..') / 'config' / 'example_SSc.yaml'))

# Cohort labels and column names used below
status_col = config.require('cohort', 'status_column')
meta_id_col = config.require('cohort', 'metadata_sample_column')
case_label = config.require('cohort', 'case_label')
control_label = config.require('cohort', 'control_label')

exclusions = load_exclusion_lists(config)
annovar_vcf_root = config.path('mutation', 'annovar_dir')
today = datetime.now().strftime('%m_%d_%y')

In [ ]:
# Load raw metadata directly to get the full sample list and case/control IDs for processing
raw_meta_df = pd.read_csv(config.path('inputs', 'metadata'), dtype={meta_id_col: str})
raw_meta_df = clean_dataframe(raw_meta_df)
raw_meta_df[status_col] = raw_meta_df[status_col].str.strip()

case_ids = raw_meta_df.loc[raw_meta_df[status_col] == case_label][meta_id_col].unique().tolist()
control_ids = raw_meta_df.loc[raw_meta_df[status_col] == control_label][meta_id_col].unique().tolist()
all_sample_ids = case_ids + control_ids

# Build a design dataframe from all samples, applying the configured exclusions
# (panel-of-normals samples and samples flagged for multiple antibody positivity)
raw_design_df = raw_meta_df.loc[raw_meta_df[meta_id_col].isin(all_sample_ids)].copy()
raw_design_df = raw_design_df.rename(columns={meta_id_col: 'sample_id'})
if config.get('exclusions', 'exclude_multi_antibody'):
    raw_design_df = raw_design_df[~raw_design_df['sample_id'].isin(exclusions['multi_antibody'])]
if config.get('exclusions', 'exclude_panel_of_normals'):
    raw_design_df = raw_design_df[~raw_design_df['sample_id'].isin(exclusions['panel_of_normals'])]

print("Number of cases and controls in the metadata, and samples remaining after exclusions:")
print(f"Cases: {len(case_ids)}")
print(f"Controls: {len(control_ids)}")
print(f"Samples after exclusions: {len(raw_design_df)}")

passing_ids_out = config.path('mutation', 'passing_sample_ids')
passing_snv_out = config.path('mutation', 'snv_table')

print("Input folder and output files for this notebook:")
print(f"Multianno input dir: {annovar_vcf_root}")
print(f"Passing IDs output: {passing_ids_out}")
print(f"SNV df output: {passing_snv_out}")

In [ ]:
# Process genomic data from multianno files
print("Reading the multianno table of each sample and classifying samples by QC status:")
results = vcf_funcs.process_genomic_data(
    raw_design_df, str(annovar_vcf_root), vcf_funcs.extract_dp, vcf_funcs.extract_ad_sb,
    file_prefix=config.require('mutation', 'annovar_file_prefix'),
    file_suffix=config.require('mutation', 'annovar_file_suffix')
)

In [ ]:
# Report the number of samples in each QC class
print("Number of samples in each QC class:")
print(f"  Empty samples: {len(results['empty_samples'])}")
print(f"  Truncated samples: {len(results['truncated_samples'])}")
print(f"  All chromosome samples: {len(results['all_chr_samples'])}")
print(f"  Passing samples: {len(results['passing_samples'])}")

In [ ]:
# Extract passing sample analysis
print("Building SNV count tables and summary statistics for the passing samples:")
analysis_results = vcf_funcs.extract_passing_sample_analysis(
    results, case_ids, control_ids
)

print(f"\nFound {len(analysis_results['snv_passing_ids'])} passing samples for downstream analysis")

In [ ]:
# Export truncated samples for realignment or exclusion
print("Writing the list of empty and truncated samples:")
truncated_filepath = vcf_funcs.export_truncated_samples(results, config.output_dir('mutation', 'sample_qc_dir'), today)
print(f"Truncated samples exported to: {truncated_filepath}")

In [ ]:
# Save passing sample IDs and SNV dataframe
print("Writing the passing sample IDs and the SNV count table:")
sample_ids_file = vcf_funcs.save_passing_sample_ids(analysis_results, passing_ids_out)
snv_data_file = vcf_funcs.save_passing_snv_df(analysis_results, passing_snv_out)
print(f"\nCreated files:")
print(f"  Sample IDs: {sample_ids_file}")
print(f"  SNV data: {snv_data_file}")

In [ ]:
# Plot number of Mutect2 PASS variants for each chromosome (normalized)
n_truncated = len(results['truncated_samples'])
n_passing = len(results['passing_samples'])

print("Figure: mean number of variants per chromosome per sample, in truncated samples (left) "
      "and passing samples (right)")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 6))

(analysis_results['chr_counts'] / n_truncated).plot(kind='bar', color='skyblue', ax=ax1)
ax1.set_title(f'Avg Variants per Chromosome per Sample (Truncated, n={n_truncated})')
ax1.set_xlabel('Chromosome')
ax1.set_ylabel('Mean Variant Count')
ax1.tick_params(axis='x', rotation=45)
ax1.grid(axis='y')

(analysis_results['chr_count_passing'] / n_passing).plot(kind='bar', color='lightgreen', ax=ax2)
ax2.set_title(f'Avg Variants per Chromosome per Sample (Passing, n={n_passing})')
ax2.set_xlabel('Chromosome')
ax2.set_ylabel('Mean Variant Count')
ax2.tick_params(axis='x', rotation=45)
ax2.grid(axis='y')

plt.tight_layout()
plt.show()